In [1]:
import os
import random
import shutil
from pathlib import Path

In [2]:
def split_random(dataset_path, nome_saida, seed):
    
    random.seed(seed)
    
    # Caminhos de entrada e saída
    dataset_path = Path(dataset_path)
    destino_base = Path(nome_saida)

    # Criação das pastas de saída: train, val, test
    destino_base.mkdir(parents=True, exist_ok=True)
    train_dir = destino_base / "train"
    val_dir = destino_base / "val"
    test_dir = destino_base / "test"
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    # Itera sobre cada subpasta (classe) no dataset
    for class_dir in dataset_path.iterdir():
        if class_dir.is_dir():
            # Criação das pastas para cada classe
            class_train = train_dir / class_dir.name
            class_val = val_dir / class_dir.name
            class_test = test_dir / class_dir.name
            class_train.mkdir(parents=True, exist_ok=True)
            class_val.mkdir(parents=True, exist_ok=True)
            class_test.mkdir(parents=True, exist_ok=True)

            # Lista de imagens na classe
            imagens = list(class_dir.glob("*.*"))  # Para imagens de qualquer formato

            # Embaralha as imagens aleatoriamente
            random.shuffle(imagens)

            # Calcula as quantidades para train, val e test
            total_imagens = len(imagens)
            train_size = int(total_imagens * 0.7)
            val_size = int(total_imagens * 0.15)
            test_size = total_imagens - train_size - val_size  # Resto das imagens

            # Divide as imagens
            train_images = imagens[:train_size]
            val_images = imagens[train_size:train_size + val_size]
            test_images = imagens[train_size + val_size:]

            # Move as imagens para os diretórios correspondentes
            for img in train_images:
                shutil.copy(str(img), str(class_train / img.name))
            for img in val_images:
                shutil.copy(str(img), str(class_val / img.name))
            for img in test_images:
                shutil.copy(str(img), str(class_test / img.name))

    print(f"\n Dataset dividido e salvo em: {destino_base}")

In [3]:
def add_syn_random(dataset1_base, dataset2, dataset3_path, porcentagem, seed, verbose=False):

    random.seed(seed)
    
    dataset1_base = Path(dataset1_base)
    dataset2 = Path(dataset2)
    dataset3_path = Path(dataset3_path)

    dataset1_train = dataset1_base / "train"
    dataset1_val = dataset1_base / "val"
    dataset1_test = dataset1_base / "test"

    train_dir = dataset3_path / "train"
    val_dir = dataset3_path / "val"
    test_dir = dataset3_path / "test"

    # Verifica se diretórios existem
    if not dataset1_train.exists():
        raise FileNotFoundError("Pasta 'train' não encontrada em dataset1")

    # Criação das pastas de saída
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    def copiar_imagens(src_dir, dest_dir):
        for class_dir in src_dir.iterdir():
            if class_dir.is_dir():
                dest_class = dest_dir / class_dir.name
                dest_class.mkdir(parents=True, exist_ok=True)
                for img_path in class_dir.glob("*.*"):
                    shutil.copy(str(img_path), str(dest_class / img_path.name))

    # Copiar dataset1 (original)
    copiar_imagens(dataset1_train, train_dir)
    copiar_imagens(dataset1_val, val_dir)
    copiar_imagens(dataset1_test, test_dir)

    # Coletar imagens já presentes no train do dataset3 para evitar duplicação
    imagens_existentes = set()
    for class_dir in train_dir.iterdir():
        if class_dir.is_dir():
            imagens_existentes.update(img.name for img in class_dir.glob("*.*"))

    # Copiar as imagens do dataset2 para o dataset3 (train)
    imagens_copiadas = 0

    # Passa pelas subpastas (classes) do dataset2
    for class_dir in Path(dataset2).iterdir():
        if class_dir.is_dir():
            destino = train_dir / class_dir.name
            destino.mkdir(parents=True, exist_ok=True)

            # Contabiliza o número de imagens da classe no dataset1
            imagens_classe_train = list(dataset1_train.glob(f"{class_dir.name}/*.*"))
            total_imagens_classe = len(imagens_classe_train)

            # Calcula o número de imagens que será copiado dessa classe (porcentagem)
            imagens_a_copiar = int(total_imagens_classe * (porcentagem / 100))
            print(f"Classe: {class_dir.name}, Imagens a copiar: {imagens_a_copiar}")

            # Pegando as imagens do dataset2 para essa classe
            imagens = list(class_dir.glob("*.*"))
            random.shuffle(imagens)  # Embaralha as imagens para aleatoriedade

            copiados_na_classe = 0
            imagens_copiadas = 0
            for img_path in imagens:
                if imagens_copiadas >= imagens_a_copiar:
                    break
                if img_path.name not in imagens_existentes:
                    shutil.copy(str(img_path), str(destino / img_path.name))
                    imagens_existentes.add(img_path.name)
                    imagens_copiadas += 1
                    copiados_na_classe += 1
                    if verbose:
                        print(f"[{class_dir.name}] Copiada: {img_path.name}")

            # Se já copiou a quantidade desejada da classe, continua para a próxima classe
            if copiados_na_classe >= imagens_a_copiar:
                continue
                
    return imagens_copiadas*10

In [4]:
seeds = [1111, 1313, 1515, 1717, 1919]
percents = [10, 30, 70, 100, 200, 300, 500, 1000]

for seed in seeds:
    #original_ds = 'DATASET5000'
    print(f"Criando datasets para seed:    {seed}")
    splited_ds  = f'DS_REAL_SPLITED_{seed}'
    #split_random(original_ds, splited_ds, seed)

    for percent in percents:
        syn_ds = 'DDPM-FILTRO-X4'
        syn_added_ds = f'DS_REAL_SPLITED_{seed}_SYN_{percent}'
        num_added_img = add_syn_random(splited_ds, syn_ds, syn_added_ds, percent, seed, verbose=False)
        print(f'\n Número de imagens adicionadas no DS {syn_added_ds} = {num_added_img} \n')

Criando datasets para seed:    1111
Classe: BULKCARRIER, Imagens a copiar: 35
Classe: CONTAINERSHIP, Imagens a copiar: 35
Classe: GENERALCARGO, Imagens a copiar: 35
Classe: OILPRODUCTSTANKER, Imagens a copiar: 35
Classe: PASSENGERSSHIP, Imagens a copiar: 35
Classe: TANKER, Imagens a copiar: 35
Classe: TRAWLER, Imagens a copiar: 35
Classe: TUG, Imagens a copiar: 35
Classe: VEHICLESCARRIER, Imagens a copiar: 35
Classe: YACHT, Imagens a copiar: 35

 Número de imagens adicionadas no DS DS_REAL_SPLITED_1111_SYN_10 = 350 

Classe: BULKCARRIER, Imagens a copiar: 105
Classe: CONTAINERSHIP, Imagens a copiar: 105
Classe: GENERALCARGO, Imagens a copiar: 105
Classe: OILPRODUCTSTANKER, Imagens a copiar: 105
Classe: PASSENGERSSHIP, Imagens a copiar: 105
Classe: TANKER, Imagens a copiar: 105
Classe: TRAWLER, Imagens a copiar: 105
Classe: TUG, Imagens a copiar: 105
Classe: VEHICLESCARRIER, Imagens a copiar: 105
Classe: YACHT, Imagens a copiar: 105

 Número de imagens adicionadas no DS DS_REAL_SPLITED_